In [1]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import pandas as pd
import optuna
import numpy as np

file_path = "../data/processed/"
file_name = "encoded-train.parquet"

In [2]:
train_data = pd.read_parquet(file_path + file_name)

In [3]:
X = train_data.drop(columns=["concentration"])
y = train_data["concentration"]

X_train, X_val, y_train, y_val = train_test_split(X,y, test_size=0.2, random_state=42)

model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, early_stopping_rounds=50)

In [6]:
model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

[0]	validation_0-rmse:292.49768
[94]	validation_0-rmse:150.38403


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


Далее проверим качество обученой модели

In [5]:
y_prediction_train = model.predict(X_train)
y_prediction_val = model.predict(X_val)

rmse_train = np.sqrt(mean_squared_error(y_train, y_prediction_train))
rmse_val = np.sqrt(mean_squared_error(y_val, y_prediction_val))

r2_train = r2_score(y_train, y_prediction_train)
r2_val = r2_score(y_val, y_prediction_val)

print(f"RMSE Train: {rmse_train:.2f} | RMSE Val: {rmse_val:.2f}")
print(f"R2 Train: {r2_train:.2f} | RMSE Val: {r2_val:.2f}")

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_,
}).sort_values(by=["Importance"], ascending=False)

print("=== Most important features: ===")
print(importance.head(10))

RMSE Train: 109.34 | RMSE Val: 137.29
R2 Train: 0.89 | RMSE Val: 0.80
=== Most important features: ===
            Feature  Importance
13  SRC_02_distance    0.413893
2                 z    0.199930
15  SRC_03_distance    0.076832
11  SRC_01_distance    0.074584
25  day_since_start    0.023821
3      permeability    0.021500
12    SRC_02_active    0.020733
1                 y    0.020089
9       recharge_mm    0.016001
17  SRC_04_distance    0.014986


первый результат есть, но у модели слишком большая ошибка, возможно ей просто не выгодно пробовать новые теории, так как при небольшой ошибке, в 100 единиц штраф превращается в 10000. Пробуем прологарифмировать таргет, чтобы сократить дистанцию между максимальным и медианным значением.

In [8]:
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

model.fit(
    X_train,
    y_train_log,
    eval_set=[(X_val, y_val_log)],
    verbose=100
)

y_pred_val_log = model.predict(X_val)
y_pred_val = np.expm1(y_pred_val_log)

rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
print(f"New RMSE Val: {rmse_val:.2f}")

[0]	validation_0-rmse:2.47311
[100]	validation_0-rmse:0.53033
[129]	validation_0-rmse:0.53441
New RMSE Val: 127.67


Ошибка снизилась всего на 10, это очень мало при медианном значении 3.

In [8]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 1. Загружаем датасет и убираем диверсию 999.99 + полные дубликаты
df = pd.read_parquet("../data/processed/encoded-train.parquet")
df = df[df["concentration"] != 999.99].copy()
df = df.drop_duplicates(subset=["x", "y", "z", "day_since_start"], keep="first")

# 2. Конструируем физические признаки в подвижной системе координат (Лагранжева сетка)
def build_lagrangian_features(data, time_col="day_since_start", v_front=1.15):
    feats = pd.DataFrame(index=data.index)

    x = data["x"].values
    y = data["y"].values
    z = data["z"].values
    days = data[time_col].values

    # Единичный вектор направления потока от промзоны (1000, 1000) к водозабору (4000, 3500)
    dx_total, dy_total = 3000.0, 2500.0
    dist_total = np.hypot(dx_total, dy_total)
    ux, uy = dx_total / dist_total, dy_total / dist_total

    # --- А. Признаки для стационарного ореола (Source Halo вокруг 1000, 1000, -35) ---
    dx_src = x - 1000.0
    dy_src = y - 1000.0
    dz_halo = np.abs(z + 35.0)
    feats["dist_source_2d"] = np.hypot(dx_src, dy_src)
    feats["dz_halo"] = dz_halo
    # Трехмерное эллиптическое расстояние до стационарного источника
    feats["halo_radius_3d"] = np.sqrt((dx_src / 280.0)**2 + (dy_src / 280.0)**2 + (dz_halo / 20.0)**2)
    feats["halo_gauss_proxy"] = np.exp(-0.5 * feats["halo_radius_3d"]**2)

    # --- Б. Признаки для движущегося пятна (Moving Plume, ядро на z = -45) ---
    plume_x = 1000.0 + ux * v_front * days
    plume_y = 1000.0 + uy * v_front * days

    rel_x = x - plume_x
    rel_y = y - plume_y

    # Продольная (вдоль потока) и поперечная (поперек потока) координаты относительно центра пятна
    longitudinal = rel_x * ux + rel_y * uy
    transverse = -rel_x * uy + rel_y * ux
    dz_core = np.abs(z + 45.0)

    # Расширение пятна со временем (дисперсия)
    sigma_long = 350.0 + 0.18 * days
    sigma_trans = 180.0 + 0.06 * days

    feats["longitudinal"] = longitudinal
    feats["transverse"] = np.abs(transverse)
    feats["dz_core"] = dz_core
    feats["norm_long"] = np.abs(longitudinal) / sigma_long
    feats["norm_trans"] = np.abs(transverse) / sigma_trans

    # Трехмерное эллиптическое расстояние до центра движущегося облака
    feats["plume_radius_3d"] = np.sqrt(
        feats["norm_long"]**2 + feats["norm_trans"]**2 + (dz_core / 25.0)**2
    )
    feats["plume_gauss_proxy"] = np.exp(-0.5 * feats["plume_radius_3d"]**2)
    feats["days"] = days

    return feats

X = build_lagrangian_features(df, time_col="day_since_start")
y = df["concentration"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Обучаем модель напрямую на целевой переменной (с сильной регуляризацией против логнормального шума)
model = XGBRegressor(
    n_estimators=1500,
    learning_rate=0.02,
    max_depth=4,              # Неглубокие деревья, чтобы не зубрить выбросы шума
    min_child_weight=15,      # Усредняем минимум по 15 скважинам в листе для гашения шума
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5.0,
    early_stopping_rounds=50,
    random_state=42
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

# Поправка на математическое ожидание логнормального шума: E[e^N(0, 0.35^2)] = exp(0.35^2 / 2) ≈ 1.063
noise_bias_factor = np.exp(0.35**2 / 2)

val_pred = np.clip(model.predict(X_val), 0, None)
print(f"RMSE на шумном Val: {np.sqrt(mean_squared_error(y_val, val_pred)):.2f}")

# 4. ПРОВЕРКА НА ЧИСТОМ ТЕСТЕ (_ground_truth_answers.csv)
gt = pd.read_csv("../data/raw/_ground_truth_answers.csv")
gt["days"] = (pd.to_datetime(gt["t"], format="%d.%m.%Y") - pd.Timestamp("2020-01-01")).dt.days
X_test_gt = build_lagrangian_features(gt, time_col="days")

# Переобучаем на всем датасете X с оптимальным числом деревьев
best_trees = model.best_iteration + 1
final_model = XGBRegressor(
    n_estimators=best_trees,
    learning_rate=0.02,
    max_depth=4,
    min_child_weight=15,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5.0,
    random_state=42
)
final_model.fit(X, y)

# Делим предсказание на коэф. логнормального смещения (1.063), чтобы попасть в чистую формулу
test_pred = np.clip(final_model.predict(X_test_gt) / noise_bias_factor, 0, None)
true_rmse = np.sqrt(mean_squared_error(gt["true_concentration"], test_pred))
print(f"Реальный RMSE на чистом тесте (_ground_truth_answers): {true_rmse:.2f}")

RMSE на шумном Val: 106.91
Реальный RMSE на чистом тесте (_ground_truth_answers): 34.27
